## 05 - Monitoring and Model Registry (Skeleton)

This notebook is a placeholder for setting up:

1. Model registration with SageMaker Model Registry  
2. Model and data monitoring (drift detection, baseline checks)


## Compress model for Sagemaker

In [33]:
import os
import shutil
import tarfile

# Ensure directories exist
os.makedirs("model", exist_ok=True)
os.makedirs("registry", exist_ok=True)

# Copy lr_model.pkl to model.pkl
src_path = "model/lr_model.pkl"
dst_path = "model/model.pkl"
shutil.copyfile(src_path, dst_path)
print("Copied lr_model.pkl to model.pkl")

# Compress the pickle model into tar.gz format
with tarfile.open("registry/model.tar.gz", "w:gz") as tar:
    tar.add("model/model.pkl", arcname="model.pkl")
    tar.add("inference.py", arcname="inference.py")

print("Compressed model into registry/model.tar.gz")


Copied lr_model.pkl to model.pkl
Compressed model into registry/model.tar.gz


## Sagemaker Setup

In [34]:
from sagemaker import Session

session = Session()
bucket = session.default_bucket()
model_key = "diabetes/registry/model.tar.gz"

# Upload tar.gz to S3
model_s3_uri = session.upload_data(
    path="registry/model.tar.gz",
    bucket=bucket,
    key_prefix="diabetes/registry"
)

print("Uploaded model to:", model_s3_uri)


Uploaded model to: s3://sagemaker-us-east-1-380537322556/diabetes/registry/model.tar.gz


## Create Model Package Group

In [35]:
# Note: error output is ok if group is previously created

import boto3
import sagemaker
from sagemaker import image_uris

model_package_group_name = "ReadmissionModelGroup"

# Get the Scikit-learn container URI for current region
region = session.boto_region_name
sklearn_uri = image_uris.retrieve(framework="sklearn", region=region, version="1.0-1")

sm_client = boto3.client("sagemaker")

# Create model package group (if not exists)
try:
    sm_client.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription="Logistic Regression for hospital readmission prediction"
    )
    print("Created model package group:", model_package_group_name)
except sm_client.exceptions.ResourceInUse:
    print("Model package group already exists.")


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:15                                                                                   │
│                                                                                                  │
│   12                                                                                             │
│   13 # Create model package group (if not exists)                                                │
│   14 try:                                                                                        │
│ ❱ 15 │   sm_client.create_model_package_group(                                                   │
│   16 │   │   ModelPackageGroupName=model_package_group_name,                                     │
│   17 │   │   ModelPackageGroupDescription="Logistic Regression for hospital readmission predi    │
│   18 │   )                                                                                       │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:569 in _api_call                      │
│                                                                                                  │
│    566 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    567 │   │   │   │   )                                                                         │
│    568 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  569 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    570 │   │                                                                                     │
│    571 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    572                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1023 in _make_api_call                │
│                                                                                                  │
│   1020 │   │   │   │   "Code"                                                                    │
│   1021 │   │   │   )                                                                             │
│   1022 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1023 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1024 │   │   else:                                                                             │
│   1025 │   │   │   return parsed_response                                                        │
│   1026                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ClientError: An error occurred (ValidationException) when calling the CreateModelPackageGroup operation: Model 
Package Group already exists: arn:aws:sagemaker:us-east-1:380537322556:model-package-group/readmissionmodelgroup

## Register Model

In [36]:
# Register the model version
response = sm_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription="Logistic Regression model (C=0.01, class_weight=balanced)",
    InferenceSpecification={
        "Containers": [{
            "Image": sklearn_uri,
            "ModelDataUrl": model_s3_uri
        }],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"]
    },
    ModelApprovalStatus="PendingManualApproval"
)

print("Registered model version in:", model_package_group_name)
print("Model Package ARN:", response["ModelPackageArn"])


Registered model version in: ReadmissionModelGroup
Model Package ARN: arn:aws:sagemaker:us-east-1:380537322556:model-package/ReadmissionModelGroup/6


## Model Monitoring

This section defines the expected structure for setting up model monitoring using Amazon SageMaker and CloudWatch.

The monitoring pipeline should include:

1. **Endpoint deployment with data capture enabled**
2. **Baseline generation using validation data** (produces statistics + constraints)
3. **Scheduling a model quality monitoring job**
4. **Capturing predictions and ground truth**
5. **Alerting via CloudWatch**
6. **Monitoring bias, explainability, and infrastructure metrics**


### Monitoring S3 Paths

The following folders should be used for monitoring-related artifacts:

- `s3://{bucket}/diabetes/monitoring/capture/` – Captured inference data
- `s3://{bucket}/diabetes/monitoring/baseline/` – Baseline statistics and constraints
- `s3://{bucket}/diabetes/monitoring/groundtruth/` – Human-labeled or synthetic ground truth labels
- `s3://{bucket}/diabetes/monitoring/reports/` – Output reports from monitoring jobs


In [ ]:
# TODO: Deploy model endpoint with data capture enabled

# from sagemaker.model import Model
# from sagemaker.model_monitor import DataCaptureConfig

# capture_config = DataCaptureConfig(
#     enable_capture=True,
#     sampling_percentage=100,
#     destination_s3_uri=f"s3://{bucket}/diabetes/monitoring/capture"
# )

# model = Model(...)
# predictor = model.deploy(initial_instance_count=1, instance_type="ml.m5.large", data_capture_config=capture_config)


In [ ]:
# TODO: Run baseline processing job using validation data

# from sagemaker.model_monitor import DefaultModelMonitor

# monitor = DefaultModelMonitor(role=role, instance_count=1, instance_type="ml.m5.large")

# monitor.suggest_baseline(
#     baseline_dataset=f"s3://{bucket}/diabetes/monitoring/baseline/validation_with_labels.csv",
#     dataset_format=DatasetFormat.csv(header=True),
#     output_s3_uri=f"s3://{bucket}/diabetes/monitoring/baseline/",
#     wait=True
# )


In [ ]:
# TODO: Schedule model quality monitoring job and optionally configure CloudWatch alerts

# monitor.create_monitoring_schedule(
#     monitor_schedule_name="readmission-monitor",
#     endpoint_input=predictor.endpoint_name,
#     output_s3_uri=f"s3://{bucket}/diabetes/monitoring/reports/"
# )

# Use CloudWatch to trigger alarms on accuracy/F1/AUC drift, latency, and infra usage
